# GRPO evaluation — Base vs trained adapter

Visual summary of the held-out **NuminaMath-TIR** comparison between the base instruct model and the GRPO LoRA checkpoint.

This notebook **only reads** saved artifacts (`summary.json`, `per_example.jsonl`); it does not rerun training or evaluation.

## Experiment summary

| Item | Value |
|------|--------|
| **Base model** | `Qwen/Qwen2-0.5B-Instruct` |
| **GRPO model** | LoRA adapters at `checkpoints/Qwen2-0.5B-GRPO` |
| **Dataset** | Held-out **NuminaMath-TIR** test split |
| **Examples** | 99 (full test split; requested 100) |
| **Generation** | Greedy, `max_new_tokens=256`, identical chat template for both models |

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Resolve evaluation directory (notebook lives next to the JSON artifacts on Perlmutter).
CANDIDATES = [
    Path("/global/cfs/cdirs/m3560/Partha/GRPO/evaluation"),
    Path.cwd(),
    Path.cwd() / "evaluation",
]
EVAL_DIR = next((p for p in CANDIDATES if (p / "summary.json").is_file()), CANDIDATES[0])
PLOT_DIR = EVAL_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = EVAL_DIR / "summary.json"
PER_EXAMPLE_PATH = EVAL_DIR / "per_example.jsonl"

with SUMMARY_PATH.open() as f:
    summary = json.load(f)

rows = []
with PER_EXAMPLE_PATH.open() as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))
df = pd.DataFrame(rows)

n_examples = summary.get("n_examples", len(df))
base = summary["base"]
grpo = summary["grpo"]
base_model_id = summary.get("base_model_id") or summary.get("base_model", "Qwen/Qwen2-0.5B-Instruct")
adapter_dir = summary.get("adapter_dir") or summary.get("grpo_adapter_dir", "checkpoints/Qwen2-0.5B-GRPO")

print(f"Loaded {n_examples} examples from {PER_EXAMPLE_PATH}")
print(f"Base: {base_model_id}")
print(f"GRPO: {adapter_dir}")

## Plots

In [ ]:
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

COLORS = {"Base": "#4C72B0", "GRPO": "#DD8452"}


def grouped_bar(
    labels: list[str],
    base_vals: list[float],
    grpo_vals: list[float],
    *,
    title: str,
    ylabel: str,
    as_percent: bool = False,
    outfile: Path,
) -> None:
    x = range(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 4.5))
    b = [v * 100 if as_percent else v for v in base_vals]
    g = [v * 100 if as_percent else v for v in grpo_vals]
    ax.bar([i - width / 2 for i in x], b, width, label="Base", color=COLORS["Base"])
    ax.bar([i + width / 2 for i in x], g, width, label="GRPO", color=COLORS["GRPO"])
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylabel("%" if as_percent else ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(outfile, bbox_inches="tight")
    plt.show()


# Plot A — quality metrics (rates stored as fractions in summary.json)
grouped_bar(
    [
        "Math accuracy",
        "Format compliance",
        "Strict success",
        "Parse failure",
    ],
    [
        base["math_accuracy"],
        base["format_compliance"],
        base["strict_success"],
        base["parse_failure_rate"],
    ],
    [
        grpo["math_accuracy"],
        grpo["format_compliance"],
        grpo["strict_success"],
        grpo["parse_failure_rate"],
    ],
    title="Plot A — Quality metrics (Base vs GRPO)",
    ylabel="Rate",
    as_percent=True,
    outfile=PLOT_DIR / "metrics_comparison.png",
)

# Plot B — generation statistics (tokens + clipped rate; two panels, one PNG)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, labels, bvals, gvals, ylab, as_pct in [
    (
        axes[0],
        ["Avg tokens", "Median tokens"],
        [base["avg_generated_tokens"], base["median_generated_tokens"]],
        [grpo["avg_generated_tokens"], grpo["median_generated_tokens"]],
        "Tokens",
        False,
    ),
    (
        axes[1],
        ["Clipped @ 256"],
        [base["fraction_clipped_at_max"]],
        [grpo["fraction_clipped_at_max"]],
        "Rate (%)",
        True,
    ),
]:
    x = range(len(labels))
    width = 0.35
    b = [v * 100 if as_pct else v for v in bvals]
    g = [v * 100 if as_pct else v for v in gvals]
    ax.bar([i - width / 2 for i in x], b, width, label="Base", color=COLORS["Base"])
    ax.bar([i + width / 2 for i in x], g, width, label="GRPO", color=COLORS["GRPO"])
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylab)
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
fig.suptitle("Plot B — Generation statistics (Base vs GRPO)")
fig.tight_layout()
fig.savefig(PLOT_DIR / "generation_stats.png", bbox_inches="tight")
plt.show()

# Plot C — efficiency
grouped_bar(
    ["Runtime (s)", "Examples / sec"],
    [base["runtime_sec"], base["examples_per_sec"]],
    [grpo["runtime_sec"], grpo["examples_per_sec"]],
    title="Plot C — Evaluation efficiency (Base vs GRPO)",
    ylabel="Seconds / throughput",
    as_percent=False,
    outfile=PLOT_DIR / "efficiency.png",
)

# Plot D — token count distributions
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = max(10, min(40, df["base_gen_tokens"].max() // 5 + 1))
ax.hist(
    df["base_gen_tokens"],
    bins=bins,
    alpha=0.55,
    label="Base",
    color=COLORS["Base"],
    edgecolor="white",
)
ax.hist(
    df["grpo_gen_tokens"],
    bins=bins,
    alpha=0.55,
    label="GRPO",
    color=COLORS["GRPO"],
    edgecolor="white",
)
ax.set_xlabel("Generated tokens per example")
ax.set_ylabel("Count")
ax.set_title("Plot D — Token count distribution (Base vs GRPO)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(PLOT_DIR / "token_histogram.png", bbox_inches="tight")
plt.show()

print("Saved PNGs under:", PLOT_DIR)

## Metrics table

In [ ]:
def pct(x: float) -> str:
    return f"{100 * x:.1f}%"


table = pd.DataFrame(
    {
        "Metric": [
            "Math accuracy",
            "Format compliance",
            "Strict (correct + format)",
            "Parse failure rate",
            "Avg generated tokens",
            "Median generated tokens",
            "Clipped at 256",
            "Runtime (s)",
            "Examples / sec",
        ],
        "Base": [
            pct(base["math_accuracy"]),
            pct(base["format_compliance"]),
            pct(base["strict_success"]),
            pct(base["parse_failure_rate"]),
            f"{base['avg_generated_tokens']:.1f}",
            f"{base['median_generated_tokens']:.0f}",
            pct(base["fraction_clipped_at_max"]),
            f"{base['runtime_sec']:.1f}",
            f"{base['examples_per_sec']:.2f}",
        ],
        "GRPO": [
            pct(grpo["math_accuracy"]),
            pct(grpo["format_compliance"]),
            pct(grpo["strict_success"]),
            pct(grpo["parse_failure_rate"]),
            f"{grpo['avg_generated_tokens']:.1f}",
            f"{grpo['median_generated_tokens']:.0f}",
            pct(grpo["fraction_clipped_at_max"]),
            f"{grpo['runtime_sec']:.1f}",
            f"{grpo['examples_per_sec']:.2f}",
        ],
    }
)
display(table)

## Interpretation

- **Format compliance rose sharply under GRPO** (~1% → ~16%), showing the reward shaping taught the required `<think>` / `<answer>` template much more often than the base model.
- **Math accuracy did not improve** on this held-out slice; base and GRPO matched at ~1%, so reasoning correctness did not transfer yet at this scale.
- **Strict success (correct + formatted) stayed at 0%** — no example satisfied both math verification and full format at once.
- **GRPO completions were shorter** on average (lower mean/median token counts and fewer sequences hitting the 256-token cap).
- **Parse failure rate increased slightly for GRPO**, consistent with terse template fills that still fail `math_verify` extraction.
- **GRPO inference pass was faster** in this eval harness (shorter outputs → higher examples/sec), though total wall time includes loading both models sequentially.